In [1]:

import os

import random
random.seed(1989)
import numpy as np
np.random.seed(1989)
import tensorflow as tf
tf.reset_default_graph()
tf.set_random_seed(1989)
from keras.models import Sequential
from keras import backend as K
from keras.layers import Input, Dense, Activation, Bidirectional
from keras.layers.core import Lambda, Reshape, Masking
from keras.layers.embeddings import Embedding
from keras.layers.recurrent import GRU, LSTM
from keras.models import Model
from keras.objectives import categorical_crossentropy
from keras import regularizers
from pprint import pprint

# from sklearn import decomposition
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D

try:
    import cpickle as pickle
except:
    import pickle


Using TensorFlow backend.


In [2]:

n                       = 3
b_size                  = 82
out_size                = 82
emb_size                = 100
lstm_size_size          = 50
voc_size                = 100000
learning_rate           = 1.0001
dense_size              = 100
nb_epoch                = 1 # 60
batch_size              = 82


In [3]:

print('Build model...')

# create empty sequential model
model = Sequential()
# add an embedding layer
model.add(Embedding( input_dim=voc_size, output_dim=emb_size, mask_zero=True))
# add an LSTM layer
model.add(LSTM(lstm_size, return_sequences=False))
# model.add(Bidirectional(GRU(gru_size, return_sequences=False)))
# add an hidden MLP layer
model.add(Dense( dense_size, activation='sigmoid' ))
# add the output MLP layer
model.add(Dense( out_size, activation='softmax', name='output', use_bias=False, ))
# compile the model usig categorical crossentropy loss 
model.compile(loss='categorical_crossentropy', optimizer='sgd')

print(model.summary())



Build model...
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_1 (Embedding)      (None, None, 100)         10000000  
_________________________________________________________________
bidirectional_1 (Bidirection (None, 100)               45300     
_________________________________________________________________
dense_1 (Dense)              (None, 100)               10100     
_________________________________________________________________
output (Dense)               (None, 82)                8200      
Total params: 10,063,600
Trainable params: 10,063,600
Non-trainable params: 0
_________________________________________________________________
None



<img src="author_embeds.png", width=2000, height=60>


In [4]:

import os
import cPickle as pickle

diri = './author_data/'
fs = [ diri+f for f in os.listdir(diri) ]
print(len(fs))

# print(pickle.load(open(fs[0],'rb'))['x'].shape)

def myGenerator():
    for f in fs:
        d = pickle.load(open(f,'rb'))
        yield d['x'],d['y']


# We train (fit our data to) our model
model.fit_generator(
    myGenerator(),
    steps_per_epoch = 1000,
    nb_epoch        = nb_epoch,
    verbose         = 1,
    validation_data = None,
)




4985


/home/user/virtual_environments/my_project/venv/lib/python2.7/site-packages/ipykernel/__main__.py:23: UserWarning: The semantics of the Keras 2 argument `steps_per_epoch` is not the same as the Keras 1 argument `samples_per_epoch`. `steps_per_epoch` is the number of batches to draw from the generator at each epoch. Basically steps_per_epoch = samples_per_epoch/batch_size. Similarly `nb_val_samples`->`validation_steps` and `val_samples`->`steps` arguments have changed. Update your method calls accordingly.
/home/user/virtual_environments/my_project/venv/lib/python2.7/site-packages/ipykernel/__main__.py:23: UserWarning: Update your `fit_generator` call to the Keras 2 API: `fit_generator(<generator..., epochs=1, validation_data=None, steps_per_epoch=1000, verbose=1)`


Epoch 1/1
1000/1000 [==============================] - 329s - loss: 4.4300      

In [5]:

ws = model.layers[-1]


In [6]:

print( ws.get_weights()[0].transpose(1,0).shape )


(82, 100)
